# LLM Next-Word Probability & Temperature

This notebook demonstrates how a Large Language Model (LLM) calculates the probability distribution for the next word after the phrase **"I am doing"**.

It uses raw scores called **Logits** and passes them through a **Softmax function** scaled by **Temperature** to convert them into percentages. 

* **T < 1.0 (Low):** Sharpens the distribution. The model becomes highly predictable and confident.
* **T = 1.0 (Default):** The base mathematical probabilities.
* **T > 1.0 (High):** Flattens the distribution. The model becomes more "creative" and random.

# The softmax function
The function acts on the logic scores (enumeration) corresponding to each probable word/token. The output is the probability score, such that the sum of all probabilities is 1 (i.e. 100%)
\begin{equation}
    \sigma(\mathbf{z})_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
\end{equation}


Where:
* $\sigma(\mathbf{z})_i$ : The resulting probability for the $i$-th _probable_ output word.
* $\mathbf{z}$ : The input vector containing the raw scores (logits); each score corresponding to the _probable_ output word/token in the order.
* $z_i$ : The specific raw input value for the $i$-th word.
* $K$ : The total number of elements in the vector (or total number of  _probable_ output words).
* $e^{z_i}$ : The standard exponential function applied to the input value, ensuring strict positivity.
* $\sum_{j=1}^{K} e^{z_j}$ : The normalization term ensuring that all final output values sum exactly to $1$ (or 100%).


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# 1. Define the candidate words and their raw base scores (logits)
words = ['great', 'well', 'my', 'a', 'this', 'fine', 'okay', 'it', 'good', 'nothing']
base_logits = np.array([2.5, 2.4, 2.0, 1.8, 1.5, 1.2, 1.0, 0.8, 0.5, 0.1])

def softmax_with_temperature(logits, temperature):
    """
    Applies the softmax function scaled by a temperature parameter.
    Formula: exp(logit / T) / sum(exp(logits / T))
    """
    # Divide logits by temperature
    scaled_logits = logits / temperature
    
    # Subtract max for numerical stability (prevents overflow during exp)
    scaled_logits -= np.max(scaled_logits)
    
    # Exponentiate and normalize to sum to 1.0 (100%)
    exp_logits = np.exp(scaled_logits)
    probabilities = exp_logits / np.sum(exp_logits)
    
    return probabilities


In [2]:
TEMPERATURE = 0.8

# 2. Create the interactive plotting function
def plot_distribution(Temperature=1.0):
    # Calculate the new probabilities based on the slider's temperature
    probs = softmax_with_temperature(base_logits, Temperature)
    
    # Set up the plot
    plt.figure(figsize=(10, 6))
    bars = plt.bar(words, probs * 100, color='#4A90E2', edgecolor='black')
    
    # Formatting
    plt.ylim(0, 100)
    plt.ylabel('Probability (%)', fontsize=12)
    plt.xlabel('Candidate Next Words', fontsize=12)
    plt.title(f'Next-Word Probability (T = {Temperature})', fontsize=14, fontweight='bold')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Add the exact percentage text on top of each bar
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 1, 
                 f'{yval:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
        
    plt.show()

# 3. Bind the slider to the function
# Note: We allow it to go up to 3.0 to clearly show the "flattening" effect
slider = FloatSlider(value=TEMPERATURE, min=0.1, max=10.0, step=0.1, description='Temperature:', continuous_update=True)
interact(plot_distribution, Temperature=slider);

interactive(children=(FloatSlider(value=0.8, description='Temperature:', max=10.0, min=0.1), Output()), _dom_c…